# `collab-sol` utilities demo

This minimal Colab example loads the final crossing-year dataset, creates leakage-safe temporal folds, tunes an ordinary scikit-learn model, and evaluates continuous risk scores on the untouched 2024 holdout. To try another model, replace only the two functions in **Define the model**.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DATASET_PATH = Path("/content/drive/MyDrive/Team 2/data/processed/final_dataset.parquet")
RUN_DIR = Path("/content/drive/MyDrive/Team 2/runs/logistic_demo")
RUN_DIR.mkdir(parents=True, exist_ok=True)

%pip install -q "collab-sol @ git+https://github.com/helloimfrei/collab-sol.git@main"

In [ ]:
import json

import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from collab_sol import (
    generate_risk_report,
    make_expanding_temporal_split,
    tune_model,
)

## Load a small feature panel

This feature list is only a runnable demonstration, not the final feature-selection decision. Outcome columns such as `incident_count`, injuries, fatalities, and severity must never be risk features.

In [ ]:
FEATURE_COLUMNS = [
    "trainsperweek",
    "maximumtimetablespeed",
    "numberofmaintracks",
    "trafficlane",
    "highwayspeedlimit",
    "annualaveragedailytrafficcount",
    "estimatedpercenttrucks",
    "pr_total_mm_5y_mean",
    "wet_days_5y_mean",
    "freeze_thaw_days_5y_mean",
    "hot_days_5y_mean",
    "tmin_mean_c_5y_mean",
    "tmax_mean_c_5y_mean",
    "wind_mean_ms_5y_mean",
]

frame = pd.read_parquet(
    DATASET_PATH,
    columns=["crossingid", "asof_year", "incident", *FEATURE_COLUMNS],
)
display(frame.groupby("asof_year")["incident"].agg(rows="size", crashes="sum", prevalence="mean"))

## Create the temporal split

With 2019–2024 data, Optuna validates on 2020, 2021, 2022, and 2023. The utility reserves 2024 as the untouched final test year. The preprocessing pipeline is rebuilt inside every fold.

In [ ]:
X = frame[FEATURE_COLUMNS]
y = frame["incident"].astype(int)
split = make_expanding_temporal_split(frame, year_column="asof_year")

def make_preprocessor():
    return make_pipeline(
        SimpleImputer(strategy="median", add_indicator=True),
        StandardScaler(),
    )

print("Validation years:", [fold.validation_year for fold in split.folds])
print("Untouched test year:", split.holdout_year)

## Define the model

These are the only two model-specific functions required by the tuner. `build_model` can return a teammate's own class as long as it implements `fit(X, y)` and `predict_proba(X)`.

In [ ]:
def suggest_params(trial):
    return {"C": trial.suggest_float("C", 1e-3, 100, log=True)}

def build_model(params, random_state):
    return LogisticRegression(
        **params,
        class_weight="balanced",
        max_iter=300,
        random_state=random_state,
    )

In [ ]:
N_TRIALS = 2  # Smoke test only; increase for the real comparison.

study = tune_model(
    model_factory=build_model,
    suggest_params=suggest_params,
    X=X,
    y=y,
    split=split,
    preprocessor_factory=make_preprocessor,
    n_trials=N_TRIALS,
    study_name="logistic_risk",
)

study.trials_dataframe().to_csv(RUN_DIR / "optuna_trials.csv", index=False)
print("Mean validation AP:", study.best_value)
print("Best parameters:", study.best_params)

## Refit once and evaluate 2024

Optuna is finished before this cell touches the holdout. The output `risk_scores.csv` contains the continuous score used to prioritize crossings, not a yes/no prediction.

In [ ]:
development_rows = split.development_indices
test_rows = split.holdout_indices

preprocessor = make_preprocessor()
X_development = preprocessor.fit_transform(X.iloc[development_rows], y.iloc[development_rows])
X_test = preprocessor.transform(X.iloc[test_rows])

model = build_model(study.best_params, random_state=42)
model.fit(X_development, y.iloc[development_rows].to_numpy())

metrics = generate_risk_report(
    model,
    X_test,
    y.iloc[test_rows].to_numpy(),
    output_dir=RUN_DIR,
    row_ids=frame.iloc[test_rows]["crossingid"].to_numpy(),
)

run_metadata = {
    "model": "logistic_regression",
    "holdout_year": split.holdout_year,
    "features": FEATURE_COLUMNS,
    "best_params": study.best_params,
}
(RUN_DIR / "run.json").write_text(json.dumps(run_metadata, indent=2))

display(pd.DataFrame([metrics.to_dict()]))
print("Report saved to:", RUN_DIR)